In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.2.3
NumPy version: 2.1.3


In [2]:
cohort = pd.read_csv("microbiology_cultures_cohort.csv")

print("Dataset loaded successfully!")
print("Shape:", cohort.shape)

Dataset loaded successfully!
Shape: (2241050, 10)


In [3]:
print("Columns:")
for i, col in enumerate(cohort.columns):
    print(f"{i}: {col}")

print("\nFirst 5 rows:")
display(cohort.head())

Columns:
0: anon_id
1: pat_enc_csn_id_coded
2: order_proc_id_coded
3: order_time_jittered_utc
4: ordering_mode
5: culture_description
6: was_positive
7: organism
8: antibiotic
9: susceptibility

First 5 rows:


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,ordering_mode,culture_description,was_positive,organism,antibiotic,susceptibility
0,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
1,JC1713666,131300064625,697566032,2020-12-27 00:40:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
2,JC1669304,131272997044,620809641,2019-07-02 19:54:00+00:00,Inpatient,BLOOD,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
3,JC1697441,131208305006,510635146,2016-12-01 13:59:00+00:00,Inpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible
4,JC600786,131344486993,834128837,2022-11-18 19:35:00+00:00,Outpatient,URINE,1,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible


In [4]:
missing = pd.DataFrame({
    "missing_count": cohort.isna().sum(),
    "missing_percent": (cohort.isna().mean() * 100).round(2)
})

display(missing.sort_values("missing_count", ascending=False))

,missing_count,missing_percent
anon_id,0,0.0
pat_enc_csn_id_coded,0,0.0
order_proc_id_coded,0,0.0
order_time_jittered_utc,0,0.0
ordering_mode,0,0.0
culture_description,0,0.0
was_positive,0,0.0
organism,0,0.0
antibiotic,0,0.0
susceptibility,0,0.0


In [5]:
cohort["susceptibility"].value_counts()

,count
susceptibility,
Susceptible,1289258
Null,634468
Resistant,265071
Intermediate,47651
Inconclusive,2670
Synergism,1932


In [6]:
ast = cohort.loc[
    cohort["susceptibility"].isin(["Susceptible", "Resistant"]),
    [
        "anon_id",
        "pat_enc_csn_id_coded",
        "order_proc_id_coded",
        "order_time_jittered_utc",
        "ordering_mode",
        "culture_description",
        "was_positive",
        "organism",
        "antibiotic",
        "susceptibility"
    ]
].copy()

ast["target"] = (
    ast["susceptibility"] == "Susceptible"
).astype("int8")

print("Rows:", len(ast))
print("Patients:", ast["anon_id"].nunique())
print("Cultures:", ast["order_proc_id_coded"].nunique())
print("Organisms:", ast["organism"].nunique())
print("Antibiotics:", ast["antibiotic"].nunique())

print("\nTarget distribution:")
print(ast["target"].value_counts())

Rows: 1554329
Patients: 67007
Cultures: 118737
Organisms: 309
Antibiotics: 54

Target distribution:
target
1    1289258
0     265071
Name: count, dtype: int64


In [7]:
antibiotics_per_culture = (
    ast.groupby("order_proc_id_coded")
    .size()
)

print(antibiotics_per_culture.describe())

print("\nDistribution:")
print(
    antibiotics_per_culture
    .value_counts()
    .sort_index()
    .head(30)
)

count    118737.000000
mean         13.090519
std           5.970197
min           1.000000
25%           8.000000
50%          14.000000
75%          17.000000
max          69.000000
dtype: float64

Distribution:
1       613
2       538
3      1029
4      3721
5      6177
6      3205
7      6670
8      7905
9      6969
10     3432
11     9875
12     4471
13     4150
14     5805
15    10827
16     9906
17    22219
18     1048
19      631
20      654
21      736
22     1168
23     1298
24      740
25      976
26      818
27      232
28      236
29      225
30      246
Name: count, dtype: int64


In [8]:
example_culture = ast["order_proc_id_coded"].iloc[0]

display(
    ast[
        ast["order_proc_id_coded"] == example_culture
    ][
        [
            "anon_id",
            "pat_enc_csn_id_coded",
            "order_proc_id_coded",
            "order_time_jittered_utc",
            "culture_description",
            "organism",
            "antibiotic",
            "susceptibility",
            "target"
        ]
    ].sort_values("antibiotic")
)

,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,culture_description,organism,antibiotic,susceptibility,target
1149880,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Amikacin,Susceptible,1
1794487,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Ampicillin,Resistant,0
12086,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Ampicillin/Sulbactam,Susceptible,1
84541,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Aztreonam,Susceptible,1
975372,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Cefazolin,Susceptible,1
1119666,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Cefepime,Susceptible,1
22820,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Ceftazidime,Susceptible,1
1556530,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Ceftriaxone,Susceptible,1
321333,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Ciprofloxacin,Susceptible,1
0,JC2744063,131368600230,928257722,2023-12-23 22:29:00+00:00,URINE,KLEBSIELLA PNEUMONIAE,Ertapenem,Susceptible,1


In [9]:
demographics = pd.read_csv(
    "microbiology_cultures_demographics.csv"
)

prior_organism = pd.read_csv(
    "microbiology_culture_prior_infecting_organism.csv"
)

class_exposure = pd.read_csv(
    "microbiology_cultures_antibiotic_class_exposure.csv"
)

subtype_exposure = pd.read_csv(
    "microbiology_cultures_antibiotic_subtype_exposure.csv"
)

print("Demographics:", demographics.shape)
print("Prior organism:", prior_organism.shape)
print("Class exposure:", class_exposure.shape)
print("Subtype exposure:", subtype_exposure.shape)

/tmp/ipykernel_35905/3515343455.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  demographics = pd.read_csv(


Demographics: (751075, 5)
Prior organism: (1083739, 6)
Class exposure: (2710181, 8)
Subtype exposure: (5402486, 9)


In [10]:
print("\nDEMOGRAPHICS")
print(demographics.columns.tolist())
display(demographics.head())

print("\nPRIOR ORGANISM")
print(prior_organism.columns.tolist())
display(prior_organism.head())

print("\nCLASS EXPOSURE")
print(class_exposure.columns.tolist())
display(class_exposure.head())

print("\nSUBTYPE EXPOSURE")
print(subtype_exposure.columns.tolist())
display(subtype_exposure.head())


DEMOGRAPHICS
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'age', 'gender']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,age,gender
0,JC1212710,131008781669,364609652,25-34 years,Null
1,JC1218441,33052060,351667304,25-34 years,Null
2,JC1261412,131004002288,355531114,25-34 years,Null
3,JC1224853,19532426,335118697,25-34 years,Null
4,JC1242725,32587123,349578403,25-34 years,Null



PRIOR ORGANISM
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'prior_organism', 'prior_infecting_organism_days_to_culutre']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,prior_organism,prior_infecting_organism_days_to_culutre
0,JC1000055,131007833415,360359154,2009-11-27 02:52:00+00:00,Escherichia,40
1,JC1000080,131270202433,615165101,2019-07-19 22:51:00+00:00,Escherichia,2361
2,JC1000083,131013906068,384652929,2011-06-18 00:10:00+00:00,Staphylococcus,30
3,JC1000129,131354606122,876959985,2023-04-23 22:44:00+00:00,Escherichia,91
4,JC1000129,131354606122,876959985,2023-04-23 22:44:00+00:00,Escherichia,115



CLASS EXPOSURE
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'medication_category', 'medication_name', 'antibiotic_class', 'time_to_culturetime']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,medication_category,medication_name,antibiotic_class,time_to_culturetime
0,JC600474,131332758707,794295244,2022-05-13 16:38:00+00:00,RIF1,Rifampin,Ansamycin,993.0
1,JC989728,131146201135,484084816,2016-02-02 01:54:00+00:00,RIF1,Rifampin,Ansamycin,55.0
2,JC680296,131309701100,736813810,2021-07-25 23:37:00+00:00,RIF1,Rifampin,Ansamycin,1829.0
3,JC2740316,131304422274,718374179,2021-03-19 19:18:00+00:00,RIF1,Rifampin,Ansamycin,192.0
4,JC1624219,131355262042,880563445,2023-06-27 09:02:00+00:00,RIF1,Rifampin,Ansamycin,3450.0



SUBTYPE EXPOSURE
['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'medication_category', 'medication_name', 'antibiotic_subtype', 'antibiotic_subtype_category', 'medication_time_to_cultureTime']


,anon_id,pat_enc_csn_id_coded,order_proc_id_coded,order_time_jittered_utc,medication_category,medication_name,antibiotic_subtype,antibiotic_subtype_category,medication_time_to_cultureTime
0,JC661393,131323239770,764294462,2021-12-11 00:00:00+00:00,RIF2,Rifabutin,Ansamycin,ANS,31
1,JC2160561,131338576141,825909791,2022-11-09 22:15:00+00:00,RIF2,Rifabutin,Ansamycin,ANS,999
2,JC2934880,131343502227,830405835,2022-11-03 20:52:00+00:00,RIF2,Rifabutin,Ansamycin,ANS,181
3,JC902549,131312194725,730129849,2021-05-27 20:12:00+00:00,RIF2,Rifabutin,Ansamycin,ANS,1026
4,JC881542,131042033458,446700182,2014-09-18 18:54:00+00:00,SIL,Silver Sulfadiazine,Sulfonamide,SUL,123


In [11]:
keys = ["anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded"]

ast_cultures = ast[keys].drop_duplicates()

print("AST cultures:", len(ast_cultures))

demo_matches = ast_cultures.merge(
    demographics[keys].drop_duplicates(),
    on=keys,
    how="inner"
)
print("With demographics:", len(demo_matches))

prior_matches = ast_cultures.merge(
    prior_organism[keys].drop_duplicates(),
    on=keys,
    how="inner"
)
print("With prior organism:", len(prior_matches))

class_matches = ast_cultures.merge(
    class_exposure[keys].drop_duplicates(),
    on=keys,
    how="inner"
)
print("With class exposure:", len(class_matches))

subtype_matches = ast_cultures.merge(
    subtype_exposure[keys].drop_duplicates(),
    on=keys,
    how="inner"
)
print("With subtype exposure:", len(subtype_matches))

print("\nDemographics duplicate culture keys:",
      demographics.duplicated(keys).sum())

print("Prior organism duplicate culture keys:",
      prior_organism.duplicated(keys).sum())

print("Class exposure duplicate culture keys:",
      class_exposure.duplicated(keys).sum())

print("Subtype exposure duplicate culture keys:",
      subtype_exposure.duplicated(keys).sum())

AST cultures: 118737
With demographics: 118737
With prior organism: 56090
With class exposure: 72632
With subtype exposure: 82756

Demographics duplicate culture keys: 0
Prior organism duplicate culture keys: 874016
Class exposure duplicate culture keys: 2293676
Subtype exposure duplicate culture keys: 4918116


In [12]:
print("PRIOR ORGANISM")
print("Unique prior organisms:", prior_organism["prior_organism"].nunique())
print("Days to culture:")
display(prior_organism["prior_infecting_organism_days_to_culutre"].describe())

print("\nCLASS EXPOSURE")
print("Unique medication names:", class_exposure["medication_name"].nunique())
print("Unique antibiotic classes:", class_exposure["antibiotic_class"].nunique())
print("Time to culture:")
display(class_exposure["time_to_culturetime"].describe())

print("\nSUBTYPE EXPOSURE")
print("Unique medication names:", subtype_exposure["medication_name"].nunique())
print("Unique antibiotic subtypes:", subtype_exposure["antibiotic_subtype"].nunique())
print("Unique subtype categories:", subtype_exposure["antibiotic_subtype_category"].nunique())
print("Time to culture:")
display(subtype_exposure["medication_time_to_cultureTime"].describe())

print("\nTOP PRIOR ORGANISMS")
display(prior_organism["prior_organism"].value_counts().head(15))

print("\nTOP ANTIBIOTIC CLASSES")
display(class_exposure["antibiotic_class"].value_counts().head(15))

print("\nTOP ANTIBIOTIC SUBTYPES")
display(subtype_exposure["antibiotic_subtype"].value_counts().head(15))

PRIOR ORGANISM
Unique prior organisms: 16
Days to culture:


,prior_infecting_organism_days_to_culutre
count,1.083739e+06
mean,1.091839e+03
std,1.067933e+03
min,1.000000e+00
25%,2.550000e+02
50%,7.450000e+02
75%,1.619000e+03
max,5.770000e+03



CLASS EXPOSURE
Unique medication names: 90
Unique antibiotic classes: 18
Time to culture:


,time_to_culturetime
count,2.710180e+06
mean,9.517270e+02
std,1.008461e+03
min,1.000000e+00
25%,1.730000e+02
50%,5.930000e+02
75%,1.421000e+03
max,5.748000e+03



SUBTYPE EXPOSURE
Unique medication names: 89
Unique antibiotic subtypes: 25
Unique subtype categories: 25
Time to culture:


...

,count
prior_organism,
Escherichia,274323
Staphylococcus,213565
Pseudomonas,207507
Enterococcus,106145
Klebsiella,86309
Streptococcus,35915
Proteus,31445
CONS,28428
Stenotrophomonas,25211



TOP ANTIBIOTIC CLASSES


,count
antibiotic_class,
Beta Lactam,809452
Fluoroquinolone,497933
Combination Antibiotic,342133
Macrolide Lincosamide,295795
Nitrofuran,150534
Nitroimidazole,138470
Glycopeptide,137832
Ansamycin,89724
Tetracycline,81894



TOP ANTIBIOTIC SUBTYPES


,count
antibiotic_subtype,
Fluoroquinolone,1000961
Cephalosporin Gen1,852141
Beta Lactam Combo,586822
Sulfonamide Combo,536609
Macrolide,535895
Nitroimidazole,245148
Nitrofuran,243517
Glycopeptide,212590
Tetracycline,173620


In [13]:
print("was_positive in original cohort:")
print(cohort["was_positive"].value_counts(dropna=False))

print("\nwas_positive in AST subset:")
print(ast["was_positive"].value_counts(dropna=False))

print("\nCross-tabulation:")
print(
    pd.crosstab(
        cohort["was_positive"],
        cohort["susceptibility"],
        dropna=False
    )
)

print("\nExamples:")
print(
    cohort[
        ["was_positive", "culture_description", "organism", "antibiotic", "susceptibility"]
    ].head(20).to_string(index=False)
)

was_positive in original cohort:
was_positive
1    1610252
0     630798
Name: count, dtype: int64

was_positive in AST subset:
was_positive
1    1554329
Name: count, dtype: int64

Cross-tabulation:
susceptibility  Inconclusive  Intermediate    Null  Resistant  Susceptible  \
was_positive                                                                 
0                          0             0  630798          0            0   
1                       2670         47651    3670     265071      1289258   

susceptibility  Synergism  
was_positive               
0                       0  
1                    1932  

Examples:
 was_positive culture_description              organism antibiotic susceptibility
            1               URINE KLEBSIELLA PNEUMONIAE  Ertapenem    Susceptible
            1               URINE KLEBSIELLA PNEUMONIAE  Ertapenem    Susceptible
            1               BLOOD KLEBSIELLA PNEUMONIAE  Ertapenem    Susceptible
            1               URINE KLEB

In [14]:
keys = ["anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded"]

culture_table = ast[keys].drop_duplicates().reset_index(drop=True)
culture_table["culture_idx"] = np.arange(
    len(culture_table),
    dtype=np.int32
)

print("Unique cultures:", len(culture_table))
print("Culture table shape:", culture_table.shape)
print("Unique culture_idx:", culture_table["culture_idx"].nunique())

Unique cultures: 118737
Culture table shape: (118737, 4)
Unique culture_idx: 118737


In [15]:
ast = ast.merge(
    culture_table,
    on=keys,
    how="left",
    sort=False
)

print("AST shape:", ast.shape)
print("Missing culture_idx:", ast["culture_idx"].isna().sum())
print("culture_idx dtype:", ast["culture_idx"].dtype)

AST shape: (1554329, 12)
Missing culture_idx: 0
culture_idx dtype: int32


In [16]:
demo = demographics[
    keys + ["age", "gender"]
].drop_duplicates(keys)

culture_features = culture_table.merge(
    demo,
    on=keys,
    how="left",
    sort=False
)

print("Culture feature shape:", culture_features.shape)
print("Missing age:", culture_features["age"].isna().sum())
print("Missing gender:", culture_features["gender"].isna().sum())
print("\nAge values:")
print(culture_features["age"].value_counts(dropna=False).sort_index())
print("\nGender values:")
print(culture_features["gender"].value_counts(dropna=False))

Culture feature shape: (118737, 6)
Missing age: 0
Missing gender: 0

Age values:
age
18-24 years     9163
25-34 years    13293
35-44 years    11470
45-54 years    13258
55-64 years    17362
65-74 years    21096
75-84 years    19398
85-89 years     7271
above 90        6426
Name: count, dtype: int64

Gender values:
gender
0       64720
0       22727
1       17825
1       13428
Null       37
Name: count, dtype: int64


In [17]:
prior = prior_organism[
    keys + ["prior_organism", "prior_infecting_organism_days_to_culutre"]
].merge(
    culture_table[["culture_idx"] + keys],
    on=keys,
    how="inner",
    sort=False
)

prior["days"] = pd.to_numeric(
    prior["prior_infecting_organism_days_to_culutre"],
    errors="coerce"
).astype("float32")

print("Matched prior-organism rows:", len(prior))
print("Cultures with prior-organism history:", prior["culture_idx"].nunique())
print("Unique prior organisms:", prior["prior_organism"].nunique())
print("Missing days:", prior["days"].isna().sum())

Matched prior-organism rows: 360837
Cultures with prior-organism history: 56090
Unique prior organisms: 16
Missing days: 0


In [18]:
prior_base = (
    prior.groupby("culture_idx")
    .agg(
        prior_organism_count=("prior_organism", "size"),
        days_since_prior_organism=("days", "min")
    )
    .reset_index()
)

prior_counts = (
    prior.groupby(["culture_idx", "prior_organism"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

prior_counts.columns = [
    "culture_idx"
] + [
    f"prior_{str(c)}_count"
    for c in prior_counts.columns[1:]
]

prior_features = prior_base.merge(
    prior_counts,
    on="culture_idx",
    how="left",
    sort=False
)

print("Prior feature shape:", prior_features.shape)
print("Cultures represented:", prior_features["culture_idx"].nunique())
print("Feature count:", len(prior_features.columns) - 1)
print("\nMissing values:", prior_features.isna().sum().sum())

Prior feature shape: (56090, 19)
Cultures represented: 56090
Feature count: 18

Missing values: 0


In [19]:
culture_features = culture_features.merge(
    prior_features,
    on="culture_idx",
    how="left",
    sort=False
)

prior_cols = [
    c for c in prior_features.columns
    if c != "culture_idx"
]

culture_features[prior_cols] = (
    culture_features[prior_cols]
    .fillna(0)
)

print("Culture feature shape:", culture_features.shape)
print("Missing prior features:", culture_features[prior_cols].isna().sum().sum())
print("Memory usage (MB):", culture_features.memory_usage(deep=True).sum() / 1024**2)

Culture feature shape: (118737, 24)
Missing prior features: 0
Memory usage (MB): 35.981064796447754


In [20]:
del prior
del prior_features
del prior_counts
prior = None
prior_features = None
prior_counts = None

print("Prior-organism temporary objects cleared.")
print("Culture feature shape:", culture_features.shape)
print("Memory usage (MB):", culture_features.memory_usage(deep=True).sum() / 1024**2)

Prior-organism temporary objects cleared.
Culture feature shape: (118737, 24)
Memory usage (MB): 35.981064796447754


In [21]:
class_exp = class_exposure[
    keys + ["antibiotic_class", "time_to_culturetime"]
].merge(
    culture_table[["culture_idx"] + keys],
    on=keys,
    how="inner",
    sort=False
)

class_exp["days"] = pd.to_numeric(
    class_exp["time_to_culturetime"],
    errors="coerce"
).astype("float32")

print("Matched class-exposure rows:", len(class_exp))
print("Cultures with class exposure:", class_exp["culture_idx"].nunique())
print("Unique antibiotic classes:", class_exp["antibiotic_class"].nunique())
print("Missing days:", class_exp["days"].isna().sum())

Matched class-exposure rows: 535249
Cultures with class exposure: 72632
Unique antibiotic classes: 18
Missing days: 1


In [22]:
class_base = (
    class_exp.groupby("culture_idx")
    .agg(
        class_exposure_count=("antibiotic_class", "size"),
        class_days_since_last=("days", "min")
    )
    .reset_index()
)

class_base["class_days_since_last"] = (
    class_base["class_days_since_last"].fillna(0)
)

class_parts = [class_base]

for window in [30, 90, 365]:
    w = class_exp.loc[
        class_exp["days"].notna() & (class_exp["days"] <= window),
        ["culture_idx", "antibiotic_class"]
    ]

    counts = (
        w.groupby(["culture_idx", "antibiotic_class"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    counts.columns = [
        "culture_idx"
    ] + [
        f"class_{window}d_{str(c)}_count"
        for c in counts.columns[1:]
    ]

    class_parts.append(counts)

class_features = class_parts[0]

for part in class_parts[1:]:
    class_features = class_features.merge(
        part,
        on="culture_idx",
        how="left",
        sort=False
    )

class_feature_cols = [
    c for c in class_features.columns
    if c != "culture_idx"
]

class_features[class_feature_cols] = (
    class_features[class_feature_cols].fillna(0)
)

print("Class feature shape:", class_features.shape)
print("Cultures represented:", class_features["culture_idx"].nunique())
print("Feature count:", len(class_feature_cols))
print("Missing values:", class_features.isna().sum().sum())

Class feature shape: (72632, 57)
Cultures represented: 72632
Feature count: 56
Missing values: 0


In [23]:
culture_features = culture_features.merge(
    class_features,
    on="culture_idx",
    how="left",
    sort=False
)

class_cols = [
    c for c in class_features.columns
    if c != "culture_idx"
]

culture_features[class_cols] = (
    culture_features[class_cols].fillna(0)
)

print("Culture feature shape:", culture_features.shape)
print("Missing class features:", culture_features[class_cols].isna().sum().sum())
print("Memory usage (MB):", culture_features.memory_usage(deep=True).sum() / 1024**2)

Culture feature shape: (118737, 80)
Missing class features: 0
Memory usage (MB): 86.25803852081299


In [24]:
del class_exp
del class_features
del class_parts
del class_base
class_exp = None
class_features = None
class_parts = None
class_base = None

print("Class-exposure temporary objects cleared.")
print("Culture feature shape:", culture_features.shape)
print("Memory usage (MB):", culture_features.memory_usage(deep=True).sum() / 1024**2)

Class-exposure temporary objects cleared.
Culture feature shape: (118737, 80)
Memory usage (MB): 86.25803852081299


In [25]:
sub_exp = subtype_exposure[
    keys + ["antibiotic_subtype", "medication_time_to_cultureTime"]
].merge(
    culture_table[["culture_idx"] + keys],
    on=keys,
    how="inner",
    sort=False
)

sub_exp["days"] = pd.to_numeric(
    sub_exp["medication_time_to_cultureTime"],
    errors="coerce"
).astype("float32")

print("Matched subtype-exposure rows:", len(sub_exp))
print("Cultures with subtype exposure:", sub_exp["culture_idx"].nunique())
print("Unique antibiotic subtypes:", sub_exp["antibiotic_subtype"].nunique())
print("Missing days:", sub_exp["days"].isna().sum())

Matched subtype-exposure rows: 1074485
Cultures with subtype exposure: 82756
Unique antibiotic subtypes: 25
Missing days: 0


In [26]:
subtype_base = (
    sub_exp.groupby("culture_idx")
    .agg(
        subtype_exposure_count=("antibiotic_subtype", "size"),
        subtype_days_since_last=("days", "min")
    )
    .reset_index()
)

subtype_parts = [subtype_base]

for window in [30, 90, 365]:
    w = sub_exp.loc[
        sub_exp["days"] <= window,
        ["culture_idx", "antibiotic_subtype"]
    ]

    counts = (
        w.groupby(["culture_idx", "antibiotic_subtype"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    counts.columns = [
        "culture_idx"
    ] + [
        f"subtype_{window}d_{str(c)}_count"
        for c in counts.columns[1:]
    ]

    subtype_parts.append(counts)

subtype_features = subtype_parts[0]

for part in subtype_parts[1:]:
    subtype_features = subtype_features.merge(
        part,
        on="culture_idx",
        how="left",
        sort=False
    )

subtype_feature_cols = [
    c for c in subtype_features.columns
    if c != "culture_idx"
]

subtype_features[subtype_feature_cols] = (
    subtype_features[subtype_feature_cols].fillna(0)
)

print("Subtype feature shape:", subtype_features.shape)
print("Cultures represented:", subtype_features["culture_idx"].nunique())
print("Feature count:", len(subtype_feature_cols))
print("Missing values:", subtype_features.isna().sum().sum())

Subtype feature shape: (82756, 78)
Cultures represented: 82756
Feature count: 77
Missing values: 0


In [27]:
culture_features = culture_features.merge(
    subtype_features,
    on="culture_idx",
    how="left",
    sort=False
)

subtype_cols = [
    c for c in subtype_features.columns
    if c != "culture_idx"
]

culture_features[subtype_cols] = (
    culture_features[subtype_cols].fillna(0)
)

print("Culture feature shape:", culture_features.shape)
print("Missing subtype features:", culture_features[subtype_cols].isna().sum().sum())
print("Memory usage (MB):", culture_features.memory_usage(deep=True).sum() / 1024**2)

Culture feature shape: (118737, 157)
Missing subtype features: 0
Memory usage (MB): 155.55873203277588


In [28]:
del sub_exp
del subtype_features
del subtype_parts
del subtype_base
sub_exp = None
subtype_features = None
subtype_parts = None
subtype_base = None

print("Subtype-exposure temporary objects cleared.")
print("Culture feature shape:", culture_features.shape)
print("Memory usage (MB):", culture_features.memory_usage(deep=True).sum() / 1024**2)

Subtype-exposure temporary objects cleared.
Culture feature shape: (118737, 157)
Memory usage (MB): 155.55873203277588


In [29]:
from sklearn.model_selection import train_test_split

patients = ast["anon_id"].unique()

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.30,
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

train_mask = ast["anon_id"].isin(train_patients).to_numpy()
val_mask = ast["anon_id"].isin(val_patients).to_numpy()
test_mask = ast["anon_id"].isin(test_patients).to_numpy()

train_idx = np.flatnonzero(train_mask)
val_idx = np.flatnonzero(val_mask)
test_idx = np.flatnonzero(test_mask)

print("Patients:")
print("Train:", len(train_patients))
print("Validation:", len(val_patients))
print("Test:", len(test_patients))

print("\nRows:")
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))
print("Total:", len(train_idx) + len(val_idx) + len(test_idx))

print("\nPatient overlap:")
print("Train ∩ Val:", len(set(train_patients) & set(val_patients)))
print("Train ∩ Test:", len(set(train_patients) & set(test_patients)))
print("Val ∩ Test:", len(set(val_patients) & set(test_patients)))

Patients:
Train: 46904
Validation: 10051
Test: 10052

Rows:
Train: 1085420
Validation: 233224
Test: 235685
Total: 1554329

Patient overlap:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [32]:
categorical_cols = [
    "organism",
    "antibiotic",
    "culture_description",
    "ordering_mode",
    "age",
    "gender"
]

culture_categorical = culture_features[
    ["culture_idx", "age", "gender"]
].copy()

ast = ast.drop(columns=["age", "gender"], errors="ignore")

ast = ast.merge(
    culture_categorical,
    on="culture_idx",
    how="left",
    sort=False
)

category_maps = {}

for col in categorical_cols:
    values = ast.iloc[train_idx][col].astype(str).unique()
    category_maps[col] = {
        value: i + 1
        for i, value in enumerate(values)
    }

for col in categorical_cols:
    print(col, "training categories:", len(category_maps[col]))

organism training categories: 275
antibiotic training categories: 54
culture_description training categories: 3
ordering_mode training categories: 3
age training categories: 9
gender training categories: 3


In [33]:
print("AST rows:", len(ast))
print("Expected rows:", 1554329)

print("\nUnique order_proc IDs:", ast["order_proc_id_coded"].nunique())
print("Expected cultures:", 118737)

print("\nMissing culture_idx:", ast["culture_idx"].isna().sum())
print("Missing age:", ast["age"].isna().sum())
print("Missing gender:", ast["gender"].isna().sum())

print("\nSplit sizes:")
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

AST rows: 1554329
Expected rows: 1554329

Unique order_proc IDs: 118737
Expected cultures: 118737

Missing culture_idx: 0
Missing age: 0
Missing gender: 0

Split sizes:
Train: 1085420
Validation: 233224
Test: 235685


In [34]:
categorical_ids = {}

for col in categorical_cols:
    values = ast[col].astype(str)
    ids = values.map(category_maps[col]).fillna(0).astype(np.int16)
    categorical_ids[col] = ids.to_numpy()

X_cat = np.column_stack([
    categorical_ids[col]
    for col in categorical_cols
])

print("Categorical matrix shape:", X_cat.shape)
print("Categorical dtype:", X_cat.dtype)

for i, col in enumerate(categorical_cols):
    print(
        col,
        "min:", X_cat[:, i].min(),
        "max:", X_cat[:, i].max(),
        "unknown:", (X_cat[:, i] == 0).sum()
    )

Categorical matrix shape: (1554329, 6)
Categorical dtype: int16
organism min: 0 max: 275 unknown: 289
antibiotic min: 1 max: 54 unknown: 0
culture_description min: 1 max: 3 unknown: 0
ordering_mode min: 1 max: 3 unknown: 0
age min: 1 max: 9 unknown: 0
gender min: 1 max: 3 unknown: 0


In [35]:
numeric_culture_cols = [
    c for c in culture_features.columns
    if c not in [
        "culture_idx",
        "anon_id",
        "pat_enc_csn_id_coded",
        "order_proc_id_coded",
        "age",
        "gender"
    ]
    and pd.api.types.is_numeric_dtype(culture_features[c])
]

culture_num = (
    culture_features[numeric_culture_cols]
    .fillna(0)
    .astype("float32")
    .to_numpy()
)

print("Culture numerical features:", len(numeric_culture_cols))
print("Culture numerical matrix:", culture_num.shape)
print("Dtype:", culture_num.dtype)
print(
    "Memory usage (MB):",
    culture_num.nbytes / 1024**2
)

Culture numerical features: 151
Culture numerical matrix: (118737, 151)
Dtype: float32
Memory usage (MB): 68.39480209350586


In [36]:
train_culture_idx = ast.iloc[train_idx]["culture_idx"].to_numpy()

train_culture_mask = np.zeros(
    len(culture_features),
    dtype=bool
)

train_culture_mask[
    np.unique(train_culture_idx)
] = True

train_mean = culture_num[train_culture_mask].mean(axis=0)
train_std = culture_num[train_culture_mask].std(axis=0)

train_std[train_std == 0] = 1.0

culture_num = (
    (culture_num - train_mean) / train_std
).astype("float32")

print("Standardized culture matrix:", culture_num.shape)
print("Dtype:", culture_num.dtype)
print("Training mean range:", train_mean.min(), "to", train_mean.max())
print("Training std range:", train_std.min(), "to", train_std.max())
print("Constant features:", np.sum(train_std == 1.0))

Standardized culture matrix: (118737, 151)
Dtype: float32
Training mean range: 0.00013282457 to 226.06506
Training std range: 0.011523495 to 493.64047
Constant features: 0


In [37]:
y = ast["target"].to_numpy(dtype=np.float32)
culture_idx = ast["culture_idx"].to_numpy(dtype=np.int32)

print("Targets:", y.shape)
print("Culture indices:", culture_idx.shape)

print("\nTarget rates:")
print("Train:", y[train_idx].mean())
print("Validation:", y[val_idx].mean())
print("Test:", y[test_idx].mean())

print("\nCulture index range:")
print("Min:", culture_idx.min())
print("Max:", culture_idx.max())

Targets: (1554329,)
Culture indices: (1554329,)

Target rates:
Train: 0.8296429
Validation: 0.82734627
Test: 0.83072746

Culture index range:
Min: 0
Max: 118736


In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

class AntibioticResistanceDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return (
            torch.tensor(X_cat[idx], dtype=torch.long),
            torch.tensor(culture_num[culture_idx[idx]], dtype=torch.float32),
            torch.tensor(y[idx], dtype=torch.float32)
        )

train_dataset = AntibioticResistanceDataset(train_idx)
val_dataset = AntibioticResistanceDataset(val_idx)
test_dataset = AntibioticResistanceDataset(test_idx)

batch_size = 512

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("\nBatches:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

batch_cat, batch_num, batch_y = next(iter(train_loader))

print("\nFirst batch:")
print("Categorical:", batch_cat.shape)
print("Numerical:", batch_num.shape)
print("Target:", batch_y.shape)

Train samples: 1085420
Validation samples: 233224
Test samples: 235685

Batches:
Train: 2120
Validation: 456
Test: 461

First batch:
Categorical: torch.Size([512, 6])
Numerical: torch.Size([512, 151])
Target: torch.Size([512])


In [39]:
import torch.nn as nn

embedding_dims = {
    "organism": 32,
    "antibiotic": 16,
    "culture_description": 4,
    "ordering_mode": 4,
    "age": 4,
    "gender": 4
}

class ResistancePredictor(nn.Module):
    def __init__(self, category_maps, num_numeric):
        super().__init__()

        self.organism_embedding = nn.Embedding(
            len(category_maps["organism"]) + 1,
            embedding_dims["organism"]
        )

        self.antibiotic_embedding = nn.Embedding(
            len(category_maps["antibiotic"]) + 1,
            embedding_dims["antibiotic"]
        )

        self.culture_embedding = nn.Embedding(
            len(category_maps["culture_description"]) + 1,
            embedding_dims["culture_description"]
        )

        self.ordering_embedding = nn.Embedding(
            len(category_maps["ordering_mode"]) + 1,
            embedding_dims["ordering_mode"]
        )

        self.age_embedding = nn.Embedding(
            len(category_maps["age"]) + 1,
            embedding_dims["age"]
        )

        self.gender_embedding = nn.Embedding(
            len(category_maps["gender"]) + 1,
            embedding_dims["gender"]
        )

        embedding_total = sum(embedding_dims.values())

        self.network = nn.Sequential(
            nn.Linear(embedding_total + num_numeric, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.30),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.20),

            nn.Linear(128, 1)
        )

    def forward(self, categorical, numerical):
        organism = self.organism_embedding(categorical[:, 0])
        antibiotic = self.antibiotic_embedding(categorical[:, 1])
        culture = self.culture_embedding(categorical[:, 2])
        ordering = self.ordering_embedding(categorical[:, 3])
        age = self.age_embedding(categorical[:, 4])
        gender = self.gender_embedding(categorical[:, 5])

        x = torch.cat(
            [
                organism,
                antibiotic,
                culture,
                ordering,
                age,
                gender,
                numerical
            ],
            dim=1
        )

        return self.network(x).squeeze(1)

model = ResistancePredictor(
    category_maps=category_maps,
    num_numeric=culture_num.shape[1]
)

print(model)
print("\nParameters:", sum(p.numel() for p in model.parameters()))

ResistancePredictor(
  (organism_embedding): Embedding(276, 32)
  (antibiotic_embedding): Embedding(55, 16)
  (culture_embedding): Embedding(4, 4)
  (ordering_embedding): Embedding(4, 4)
  (age_embedding): Embedding(10, 4)
  (gender_embedding): Embedding(4, 4)
  (network): Sequential(
    (0): Linear(in_features=215, out_features=256, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=1, bias=True)
  )
)

Parameters: 98889


In [40]:
model.eval()

with torch.no_grad():
    sample_logits = model(
        batch_cat,
        batch_num
    )

print("Input categorical shape:", batch_cat.shape)
print("Input numerical shape:", batch_num.shape)
print("Output shape:", sample_logits.shape)
print("First 10 logits:", sample_logits[:10])
print("First 10 probabilities:", torch.sigmoid(sample_logits[:10]))

Input categorical shape: torch.Size([512, 6])
Input numerical shape: torch.Size([512, 151])
Output shape: torch.Size([512])
First 10 logits: tensor([ 0.0638,  0.0305,  0.0720, -0.0157, -0.0891, -0.0049, -0.0890,  0.0335,
         0.0367, -0.1143])
First 10 probabilities: tensor([0.5159, 0.5076, 0.5180, 0.4961, 0.4777, 0.4988, 0.4778, 0.5084, 0.5092,
        0.4715])


In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Device:", device)
print("Loss:", criterion)
print("Optimizer:", optimizer)

Device: cpu
Loss: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 1e-05
)


In [42]:
model.train()

batch_cat, batch_num, batch_y = next(iter(train_loader))

batch_cat = batch_cat.to(device)
batch_num = batch_num.to(device)
batch_y = batch_y.to(device)

optimizer.zero_grad()

logits = model(batch_cat, batch_num)
loss = criterion(logits, batch_y)

loss.backward()
optimizer.step()

print("Batch loss:", loss.item())
print("Backward pass: successful")

Batch loss: 0.7375890016555786
Backward pass: successful


In [43]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch_cat, batch_num, batch_y in train_loader:
        batch_cat = batch_cat.to(device)
        batch_num = batch_num.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = model(batch_cat, batch_num)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_y.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch_cat, batch_num, batch_y in val_loader:
            batch_cat = batch_cat.to(device)
            batch_num = batch_num.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_cat, batch_num)
            loss = criterion(logits, batch_y)

            val_loss += loss.item() * batch_y.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Train Loss: {train_loss:.4f} "
        f"- Val Loss: {val_loss:.4f}"
    )

Epoch 1/5 - Train Loss: 0.3342 - Val Loss: 0.3031
Epoch 2/5 - Train Loss: 0.2955 - Val Loss: 0.2993
Epoch 3/5 - Train Loss: 0.2901 - Val Loss: 0.2992
Epoch 4/5 - Train Loss: 0.2866 - Val Loss: 0.2979
Epoch 5/5 - Train Loss: 0.2841 - Val Loss: 0.2969


In [44]:
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

model.eval()

val_probs = []
val_targets = []

with torch.no_grad():
    for batch_cat, batch_num, batch_y in val_loader:
        batch_cat = batch_cat.to(device)
        batch_num = batch_num.to(device)

        logits = model(batch_cat, batch_num)
        probs = torch.sigmoid(logits)

        val_probs.append(probs.cpu().numpy())
        val_targets.append(batch_y.numpy())

val_probs = np.concatenate(val_probs)
val_targets = np.concatenate(val_targets)

val_auc = roc_auc_score(val_targets, val_probs)
val_auprc = average_precision_score(val_targets, val_probs)
val_accuracy = accuracy_score(
    val_targets,
    (val_probs >= 0.5).astype(np.int8)
)

print("Validation AUROC:", val_auc)
print("Validation AUPRC:", val_auprc)
print("Validation Accuracy:", val_accuracy)

Validation AUROC: 0.876820466903993
Validation AUPRC: 0.9693986563371799
Validation Accuracy: 0.8739495077693548


In [45]:
from sklearn.metrics import roc_auc_score, average_precision_score

val_results = ast.iloc[val_idx][
    ["antibiotic", "target"]
].copy()

val_results["probability"] = val_probs

rows = []

for antibiotic, group in val_results.groupby("antibiotic"):
    if group["target"].nunique() < 2:
        continue

    rows.append({
        "antibiotic": antibiotic,
        "n": len(group),
        "susceptible_rate": group["target"].mean(),
        "auroc": roc_auc_score(
            group["target"],
            group["probability"]
        ),
        "auprc": average_precision_score(
            group["target"],
            group["probability"]
        )
    })

per_antibiotic = pd.DataFrame(rows).sort_values(
    "auroc",
    ascending=False
)

print(per_antibiotic.to_string(index=False))

                   antibiotic     n  susceptible_rate    auroc    auprc
                    Linezolid  2171          0.977890 0.987527 0.999708
                  Tigecycline   397          0.962217 0.982897 0.999263
       Cephalexin/Cephalothin    19          0.842105 0.979167 0.996324
                     Amikacin 10170          0.952901 0.966976 0.998100
                    Meropenem 10517          0.973947 0.966413 0.999067
               Nitrofurantoin 12473          0.878457 0.957615 0.990779
                   Vancomycin  3530          0.967139 0.944060 0.997963
                   Penicillin  3563          0.582655 0.940178 0.962257
                  Minocycline    42          0.714286 0.908333 0.969669
                    Ertapenem  8581          0.995105 0.905959 0.999321
                    Oxacillin  1715          0.694461 0.898057 0.947204
                    Cefoxitin  6758          0.840633 0.895198 0.967856
                     Imipenem  1771          0.822699 0.889159 0

In [46]:
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

model.eval()

test_probs = []
test_targets = []

with torch.no_grad():
    for batch_cat, batch_num, batch_y in test_loader:
        batch_cat = batch_cat.to(device)
        batch_num = batch_num.to(device)

        logits = model(batch_cat, batch_num)
        probs = torch.sigmoid(logits)

        test_probs.append(probs.cpu().numpy())
        test_targets.append(batch_y.numpy())

test_probs = np.concatenate(test_probs)
test_targets = np.concatenate(test_targets)

test_auc = roc_auc_score(test_targets, test_probs)
test_auprc = average_precision_score(test_targets, test_probs)
test_accuracy = accuracy_score(
    test_targets,
    (test_probs >= 0.5).astype(np.int8)
)

print("Test AUROC:", test_auc)
print("Test AUPRC:", test_auprc)
print("Test Accuracy:", test_accuracy)

Test AUROC: 0.8810437120870448
Test AUPRC: 0.9713782468533727
Test Accuracy: 0.8764919277849672


In [47]:
test_results = ast.iloc[test_idx][
    [
        "anon_id",
        "culture_idx",
        "organism",
        "antibiotic",
        "target"
    ]
].copy()

test_results["probability_susceptible"] = test_probs

print(test_results.head(10).to_string(index=False))
print("\nRows:", len(test_results))
print(
    "Probability range:",
    test_results["probability_susceptible"].min(),
    "to",
    test_results["probability_susceptible"].max()
)

  anon_id  culture_idx              organism antibiotic  target  probability_susceptible
 JC934238            7 KLEBSIELLA PNEUMONIAE  Ertapenem       1                 0.996563
JC1112952           37 KLEBSIELLA PNEUMONIAE  Meropenem       1                 0.997162
JC3185630           42 KLEBSIELLA PNEUMONIAE Ampicillin       0                 0.001062
JC6235666           51 KLEBSIELLA PNEUMONIAE Ampicillin       0                 0.000936
 JC782114           54 KLEBSIELLA PNEUMONIAE Ampicillin       0                 0.000783
 JC893883           63 KLEBSIELLA PNEUMONIAE Ampicillin       0                 0.001876
JC2227720           64 KLEBSIELLA PNEUMONIAE Ampicillin       0                 0.000824
JC2122256           72 KLEBSIELLA PNEUMONIAE Ampicillin       0                 0.000955
JC1739630           77 KLEBSIELLA PNEUMONIAE Cefuroxime       1                 0.915980
 JC515191           87 KLEBSIELLA PNEUMONIAE Gentamicin       1                 0.977586

Rows: 235685
Probabi

In [48]:
ranked_test = test_results.sort_values(
    ["culture_idx", "probability_susceptible"],
    ascending=[True, False]
).copy()

ranked_test["rank"] = (
    ranked_test.groupby("culture_idx")
    .cumcount() + 1
)

print(
    ranked_test[
        [
            "culture_idx",
            "organism",
            "antibiotic",
            "probability_susceptible",
            "target",
            "rank"
        ]
    ].head(20).to_string(index=False)
)

 culture_idx              organism                    antibiotic  probability_susceptible  target  rank
           7 KLEBSIELLA PNEUMONIAE                     Meropenem                 0.996857       1     1
           7 KLEBSIELLA PNEUMONIAE                     Ertapenem                 0.996563       1     2
           7 KLEBSIELLA PNEUMONIAE                      Amikacin                 0.995521       1     3
           7 KLEBSIELLA PNEUMONIAE       Piperacillin/Tazobactam                 0.984602       1     4
           7 KLEBSIELLA PNEUMONIAE   Amoxicillin/Clavulanic Acid                 0.983710       1     5
           7 KLEBSIELLA PNEUMONIAE                  Levofloxacin                 0.971806       1     6
           7 KLEBSIELLA PNEUMONIAE                     Cefoxitin                 0.969090       1     7
           7 KLEBSIELLA PNEUMONIAE                 Ciprofloxacin                 0.967530       1     8
           7 KLEBSIELLA PNEUMONIAE                    Gentamicin

In [51]:
ranking_rows = []

for culture_idx, group in ranked_test.groupby("culture_idx"):
    y_true = group["target"].to_numpy()
    y_score = group["probability_susceptible"].to_numpy()

    row = {
        "culture_idx": culture_idx,
        "susceptible_rate": y_true.mean()
    }

    for k in [1, 3, 5]:
        top = y_true[:k]

        row[f"precision_at_{k}"] = top.mean()

        if len(y_true) > 1:
            row[f"ndcg_at_{k}"] = ndcg_score(
                y_true.reshape(1, -1),
                y_score.reshape(1, -1),
                k=min(k, len(y_true))
            )
        else:
            row[f"ndcg_at_{k}"] = np.nan

    ranking_rows.append(row)

ranking_eval = pd.DataFrame(ranking_rows)

for k in [1, 3, 5]:
    precision = ranking_eval[f"precision_at_{k}"].mean()
    random_baseline = ranking_eval["susceptible_rate"].mean()
    ndcg = ranking_eval[f"ndcg_at_{k}"].mean()

    print(f"Precision@{k}: {precision:.4f}")
    print(f"Random baseline: {random_baseline:.4f}")
    print(f"NDCG@{k}: {ndcg:.4f}")
    print()

Precision@1: 0.9923
Random baseline: 0.8287
NDCG@1: 0.9924

Precision@3: 0.9772
Random baseline: 0.8287
NDCG@3: 0.9879

Precision@5: 0.9517
Random baseline: 0.8287
NDCG@5: 0.9830



In [52]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    test_targets,
    test_probs
)

print("Test Brier Score:", brier)

calibration = pd.DataFrame({
    "target": test_targets,
    "probability": test_probs
})

calibration["bin"] = pd.cut(
    calibration["probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

calibration_summary = (
    calibration
    .groupby("bin", observed=True)
    .agg(
        mean_predicted=("probability", "mean"),
        observed_susceptibility=("target", "mean"),
        n=("target", "size")
    )
    .reset_index()
)

print(calibration_summary.to_string(index=False))

Test Brier Score: 0.08964980062710816
          bin  mean_predicted  observed_susceptibility      n
(-0.001, 0.1]        0.011221                 0.010854   6910
   (0.1, 0.2]        0.155556                 0.164074   1414
   (0.2, 0.3]        0.249828                 0.232441   2990
   (0.3, 0.4]        0.348650                 0.314683   2663
   (0.4, 0.5]        0.453726                 0.435944   3817
   (0.5, 0.6]        0.557340                 0.523225   7083
   (0.6, 0.7]        0.647237                 0.611897  16189
   (0.7, 0.8]        0.757492                 0.731003  21227
   (0.8, 0.9]        0.857763                 0.835763  32234
   (0.9, 1.0]        0.965051                 0.964997 141158


In [53]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "category_maps": category_maps,
        "numeric_culture_cols": numeric_culture_cols,
        "train_mean": train_mean,
        "train_std": train_std,
        "embedding_dims": embedding_dims
    },
    "antibiotic_resistance_model.pt"
)

print("Model saved successfully.")

Model saved successfully.


In [54]:
from google.colab import files

files.download("antibiotic_resistance_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>